# Project summary

This notebook collects the results I would use to explain the project. It loads the saved model, so it does not repeat the Leave-One-Out search or train a new network. The three earlier notebooks contain the full analysis.

## Data and weighted target

The four explosion engines provide NS or BH outcomes for 187 common progenitors. I code BH as 1 and NS as 0, then combine N20 and W18 with weight 0.30 each and W15 and S19.8 with weight 0.20 each. The resulting score summarizes agreement among simulations. It is not an observed fate or a calibrated physical probability.

In [6]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / "data/gold/progenitor_gold.csv").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.model import StellarCollapseMLP

gold = pd.read_csv(ROOT / "data/gold/progenitor_gold.csv")
print(f"Progenitors: {len(gold)}")
print(f"Input features: {gold.shape[1] - 2}")
print("Weighted scores:")
print(gold["target_score"].value_counts().sort_index().to_string())

Progenitors: 187
Input features: 44
Weighted scores:
target_score
0.0    74
0.2     8
0.5    20
0.7     3
0.8     7
1.0    75


## Agreement between engines

Each cell below is the percentage of progenitors for which two columns give the same NS/BH label. The weighted vote uses the 0.5 cutoff. It is a reference made from the engines, so agreement with it is not accuracy against real stars.

In [7]:
engines = ["N20", "W18", "W15", "S19.8"]
outcomes = pd.read_csv(ROOT / "data/silver/engine_outcomes.csv")
vote_rows = []
for star in gold["model"]:
    row = {}
    for engine in engines:
        engine_row = outcomes[(outcomes["model"] == star) & (outcomes["engine"] == engine)]
        row[engine] = engine_row["outcome"].iloc[0]
    vote_rows.append(row)
votes = pd.DataFrame(vote_rows)
votes["Weighted vote"] = [
    "BH" if score >= 0.5 else "NS" for score in gold["target_score"]
]

labels = engines + ["Weighted vote"]
agreement = pd.DataFrame(index=labels, columns=labels)
for row in labels:
    for column in labels:
        matches = (votes[row] == votes[column]).sum()
        agreement.loc[row, column] = f"{100 * matches / len(votes):.1f}%"
display(agreement)

,N20,W18,W15,S19.8,Weighted vote
N20,100.0%,87.7%,83.4%,94.7%,87.7%
W18,87.7%,100.0%,95.7%,85.6%,100.0%
W15,83.4%,95.7%,100.0%,81.3%,95.7%
S19.8,94.7%,85.6%,81.3%,100.0%,85.6%
Weighted vote,87.7%,100.0%,95.7%,85.6%,100.0%


W18 matches the weighted vote for all 187 progenitors. W18 also contributes to that vote, so the 100% entry does not say whether either result is physically correct.

## Structure in the input features

K-Means++ groups the 187 stars using the 44 standardized pre-collapse features. The target score is added only after the groups are formed. Group 1 has 31 BH-consensus stars among 32; all 14 stars in group 2 have NS consensus. The other groups are mixed.

In [8]:
features = gold.drop(columns=["model", "target_score"])
scaled = StandardScaler().fit_transform(features)
groups = KMeans(n_clusters=5, init="k-means++", n_init=10, random_state=42).fit_predict(scaled)
consensus = np.where(gold["target_score"] == 0, "NS", np.where(gold["target_score"] == 1, "BH", "Mixed"))
counts = pd.crosstab(pd.Series(groups, name="Group"), pd.Series(consensus, name="Outcome"))
print(counts.reindex(columns=["NS", "BH", "Mixed"], fill_value=0).to_string())

Outcome  NS  BH  Mixed
Group                 
0        39  13      6
1         0  31      1
2        14   0      0
3         2   4      2
4        19  27     29


<img src="../docs/figures/kmeans_five_clusters.png" width="650" alt="Five groups on the PCA projection">

## Final model on held-out stars

The network has 44 inputs, one hidden layer with 32 neurons, and one output score. Its architecture and 132 training epochs were chosen with Leave-One-Out on the training set. The saved checkpoint is evaluated below on the same 47 held-out stars as in the full notebook. This cell does not train the network.

In [9]:
X = gold.drop(columns=["model", "target_score"])
y = gold["target_score"]
_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
saved = torch.load(ROOT / "models/stellar_collapse_mlp.pt", map_location="cpu", weights_only=True)
columns = saved["feature_names"]
mean = np.asarray(saved["scaler_mean"], dtype=np.float32)
scale = np.asarray(saved["scaler_scale"], dtype=np.float32)
inputs = torch.tensor((X_test[columns].to_numpy(dtype=np.float32) - mean) / scale)
model = StellarCollapseMLP(len(columns), saved["hidden_sizes"])
model.load_state_dict(saved["state_dict"])
model.eval()
with torch.no_grad():
    logits = model(inputs)
    scores = torch.sigmoid(logits).numpy().ravel()
    bce = torch.nn.BCEWithLogitsLoss()(logits, torch.tensor(y_test.to_numpy(), dtype=torch.float32).reshape(-1, 1)).item()

print(f"Test stars: {len(y_test)}")
print(f"BCE: {bce:.4f}")
print(f"MAE: {mean_absolute_error(y_test, scores):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, scores)):.4f}")
print(f"R²: {r2_score(y_test, scores):.4f}")
labels = ["NS", "BH"]
real = np.where(y_test.to_numpy() >= 0.5, "BH", "NS")
predicted = np.where(scores >= saved["threshold"], "BH", "NS")
print("Confusion matrix (rows: target, columns: prediction; NS then BH):")
print(confusion_matrix(real, predicted, labels=labels))

Test stars: 47
BCE: 0.3605
MAE: 0.1998
RMSE: 0.2726
R²: 0.6376
Confusion matrix (rows: target, columns: prediction; NS then BH):
[[16  4]
 [ 4 23]]


The MAE is about 0.20 on a score from 0 to 1. Using 0.5 only as an extra classification check, 39 of 47 labels agree with the weighted target. The continuous score remains the main result.

<img src="../docs/figures/test_predictions.png" width="520" alt="Predicted and target scores">

## Which inputs affected the score?

This SHAP plot uses the 47 test stars. Each dot is one star. A dot to the right raises the model's BH score relative to its reference; a dot to the left lowers it. Blue means a low feature value and pink a high one. Entropy at 1.5 solar masses has the largest mean absolute effect in this set, followed by entropy at 2 solar masses. The directions differ between stars. This describes the model's behavior, not a physical cause.

<img src="../docs/figures/feature_importance_global.png" width="700" alt="SHAP feature effects">

## Predict from a CSV

Change `INPUT_CSV` to your file and run the next cell. The CSV needs the 44 feature columns used in Gold; `model` is optional and `target_score` is not needed. The cell loads the saved model and scaler, writes a CSV with scores and NS/BH labels, and shows the first five rows. The default input is Gold only to demonstrate the command; those rows include training stars and must not be used as a new test result.

In [10]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / "models/stellar_collapse_mlp.pt").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.model import StellarCollapseMLP

INPUT_CSV = ROOT / "data/gold/progenitor_gold.csv"  # Change this path for another CSV
OUTPUT_CSV = ROOT / "data/predictions/notebook_predictions.csv"

data = pd.read_csv(INPUT_CSV)
saved = torch.load(ROOT / "models/stellar_collapse_mlp.pt", map_location="cpu", weights_only=True)
columns = saved["feature_names"]
missing = [name for name in columns if name not in data.columns]
if missing:
    raise ValueError(f"Missing feature columns: {missing}")
values = data[columns].apply(pd.to_numeric, errors="raise")
if values.isna().any().any():
    raise ValueError("Feature columns contain missing values")
mean = np.asarray(saved["scaler_mean"], dtype=np.float32)
scale = np.asarray(saved["scaler_scale"], dtype=np.float32)
inputs = torch.tensor((values.to_numpy(dtype=np.float32) - mean) / scale)
model = StellarCollapseMLP(len(columns), saved["hidden_sizes"])
model.load_state_dict(saved["state_dict"])
model.eval()
with torch.no_grad():
    scores = torch.sigmoid(model(inputs)).numpy().ravel()
result = pd.DataFrame({
    "predicted_score": scores,
    "predicted_class": np.where(scores >= saved["threshold"], "BH", "NS"),
})
if "model" in data.columns:
    result.insert(0, "model", data["model"])
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(result)} predictions to {OUTPUT_CSV}")
print(result.head().to_string(index=False))

Saved 187 predictions to /Users/fedemoro/Documents/stellar-collapse-classification/data/predictions/notebook_predictions.csv
 model  predicted_score predicted_class
s12.25         0.000228              NS
 s12.5         0.000416              NS
s12.75         0.000698              NS
 s13.0         0.001300              NS
 s13.1         0.001762              NS
